# Landsat Scene Coverage Map

Discover all Landsat scenes stored in S3, extract their bounding boxes via rasterio, and plot where each scene is located on a map.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "exploratory_analyses" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import boto3
import rasterio
from rasterio.warp import transform_bounds
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

## 1. Discover all scene prefixes in S3

In [2]:
S3_BUCKET = "allotrope-raw-data-india"
S3_REGION = "ap-south-1"

s3 = boto3.client("s3", region_name=S3_REGION)
paginator = s3.get_paginator("list_objects_v2")

# List all scene folders under landsat/
scene_prefixes = []
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix="landsat/", Delimiter="/", PaginationConfig={"PageSize": 500}):
    for folder in page.get("CommonPrefixes", []):
        scene_prefixes.append(folder["Prefix"])

print(f"Found {len(scene_prefixes)} scenes in S3")

Found 2250 scenes in S3


## 2. Extract bounding boxes from each scene's ST_B10 TIF

For each scene, find the `ST_B10.TIF` key and open it directly from S3 via rasterio's GDAL virtual filesystem. We only read the header (no pixel data), so this is fast.

In [3]:
from tqdm import tqdm

scene_bboxes = []  # List of {"scene_id": ..., "bbox": [min_lon, min_lat, max_lon, max_lat]}

for prefix in tqdm(scene_prefixes, desc="Extracting bounding boxes"):
    # List objects under this scene prefix and find ST_B10
    objects = s3.list_objects(Bucket=S3_BUCKET, Prefix=prefix)
    b10_key = None
    for obj in objects.get("Contents", []):
        if "ST_B10" in obj["Key"]:
            b10_key = obj["Key"]
            break

    if b10_key is None:
        print(f"  WARNING: No ST_B10 found in {prefix}, skipping")
        continue

    # Open directly from S3 via GDAL vsis3 — only reads header metadata
    s3_path = f"s3://{S3_BUCKET}/{b10_key}"
    try:
        with rasterio.open(s3_path) as src:
            min_lon, min_lat, max_lon, max_lat = transform_bounds(
                src.crs, "EPSG:4326", *src.bounds
            )
            scene_id = prefix.rstrip("/").split("/")[-1]
            scene_bboxes.append({
                "scene_id": scene_id,
                "bbox": [min_lon, min_lat, max_lon, max_lat],
            })
    except Exception as e:
        print(f"  ERROR reading {b10_key}: {e}")

print(f"\nExtracted bounding boxes for {len(scene_bboxes)} / {len(scene_prefixes)} scenes")

Extracting bounding boxes:  72%|███████▏  | 1616/2250 [05:35<01:07,  9.39it/s]

Extracting bounding boxes:  72%|███████▏  | 1622/2250 [05:36<00:39, 16.08it/s]

Extracting bounding boxes:  72%|███████▏  | 1625/2250 [05:36<00:33, 18.61it/s]

Extracting bounding boxes:  77%|███████▋  | 1740/2250 [05:58<00:54,  9.38it/s]

Extracting bounding boxes: 100%|██████████| 2250/2250 [07:42<00:00,  4.86it/s]


Extracted bounding boxes for 2235 / 2250 scenes


## 3. Plot scene footprints on an interactive map

Each rectangle is one Landsat scene footprint overlaid on OpenStreetMap tiles via Folium. Click a rectangle to see the scene ID. The dashed rectangle is the India search bounding box from the USGS query.

In [8]:
import folium

# Center the map on the middle of the India search region
india_bb = [68.17, 7.96, 97.40, 35.49]  # [min_lon, min_lat, max_lon, max_lat]
center_lat = (india_bb[1] + india_bb[3]) / 2
center_lon = (india_bb[0] + india_bb[2]) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=5, tiles="OpenStreetMap")

# India search bounding box as a dashed rectangle
folium.Rectangle(
    bounds=[[india_bb[1], india_bb[0]], [india_bb[3], india_bb[2]]],
    color="black",
    weight=2,
    dash_array="10",
    fill=False,
    popup="India Search Region",
).add_to(m)

# Each scene footprint as a clickable rectangle
for entry in scene_bboxes:
    mn_lon, mn_lat, mx_lon, mx_lat = entry["bbox"]
    folium.Rectangle(
        bounds=[[mn_lat, mn_lon], [mx_lat, mx_lon]],
        color="steelblue",
        weight=1,
        fill=True,
        fill_color="steelblue",
        fill_opacity=0.25,
        popup=entry["scene_id"],
    ).add_to(m)

m.save("landsat_scene_map.html")

## 4. Scene summary table

In [5]:
# Print a summary of all scenes and their approximate locations
print(f"{'Scene ID':<60} {'Center Lat':>10} {'Center Lon':>10}")
print("-" * 82)
for entry in sorted(scene_bboxes, key=lambda e: (e["bbox"][1] + e["bbox"][3]) / 2, reverse=True):
    mn_lon, mn_lat, mx_lon, mx_lat = entry["bbox"]
    clat = (mn_lat + mx_lat) / 2
    clon = (mn_lon + mx_lon) / 2
    print(f"{entry['scene_id']:<60} {clat:>10.3f} {clon:>10.3f}")

Scene ID                                                     Center Lat Center Lon
----------------------------------------------------------------------------------
LC91490352024033LGN00                                            36.035     75.195
LC91490352025147LGN00                                            36.035     75.195
LC91490352025115LGN00                                            36.035     75.202
LC91490352024289LGN00                                            36.035     75.217
LC91490352025243LGN00                                            36.035     75.229
LC91490352024273LGN00                                            36.035     75.232
LC91530352024077LGN00                                            36.035     68.991
LC91530352024157LGN00                                            36.035     69.011
LC91530352024365LGN00                                            36.035     69.045
LC91530352025303LGN00                                            36.035     69.062
LC91